In [23]:
from config import *
import Minimal_Model_1D as mm1D
import Minimal_Model_2D as mm2D
import Operators2D as O2d

In [24]:
####### First writing up a function to calculate wavefunctions #############

#t1 = time.time()

#print('t1 = ', t1)


DELTA = 0.165      # in meV
#MEFF_SC = 0.05     # Ge 
#MEFF_SM = 0.05     # proximitized Ge
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs

LY = 100                            # in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC
print('confinement energy:', OMEGA , 'meV')


### Keeping it same so as to match with previous data 

MU_SM = 0.8 + OMEGA/2        ## As per Andreev Aprroximation chemical pot >> superconducting gap 
MU_SC = 0.8 + OMEGA/2        ## constant potential over reference level across all widths

K = 4

#AX = 0.25*math.sqrt(4*hbm/MEFF_SM/OMEGA)
#AY = 0.25*math.sqrt(4*hbm/MEFF_SM/OMEGA)

#print(AX, AY)

AX = 5
AY = 5 

NSC = 500
NSM = 60
M = 100
N = 2*NSC + NSM
PHI = 0.5*np.pi 

ALPHAT = 0.1                       ### low spin splitting to just break degeneracy

F2d = mm2D.twoD_Minimal_Model(N,NSM, NSC,M,AX ,AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM)

H2d = F2d.H_sys(PHI, so = 'li')

print(H2d.shape)

E2d, V2d = eigsh(H2d ,k=K,sigma=0, return_eigenvectors=True)

E2d = np.sort(E2d)

print(E2d)

print(np.unique(E2d))

confinement energy: 1.3263440267269595 meV
(424000, 424000)
[-0.07559759 -0.07559759  0.07559759  0.07559759]
[-0.07559759 -0.07559759  0.07559759  0.07559759]


In [26]:
V2d.shape

(424000, 4)

In [34]:
np.linalg.norm(V2d[:,2])

1.0000000000000002

In [36]:
A = np.array(V2d[:,2])

In [39]:
np.sum(np.abs(A)**4)

4.472772393870838e-05

In [27]:
WV1 = V2d[:,2].reshape((2,2,M*N))
WV2 = V2d[:,3].reshape((2,2,M*N))

In [28]:
print(WV1.shape)

(2, 2, 106000)


In [32]:
print(np.sum(WV1[:,:,:]**2))

(-0.009171326497884784-0.004934603146311412j)


Calculating variation of a correlation function wrt wavefunctions

In [47]:
def Correlation_2D(A, n, ax, ay, i, j, dl=1):

    """
        Returns correlation function between point labelled by i, j on a n X m lattice
        
        Args:
            A (float) : normalizing ocnstant for correlation function
            i, j (int) : index of two poins between which correlation function needs to be calculated
            dl (float) : decay length of correlation function 
        
    """

    x_dis = ((i%n) - (j%n))*ax   ## x distance between two points on lattice with index i,j labelled in graph way 

    y_dis = ((i//n) - (j//n))*ay   ## y distance between two points on lattice with index i,j labelled in graph way 

    return A**2 * math.exp(-abs(math.sqrt(x_dis**2 + y_dis**2))/dl)

In [48]:
Correlation_2D(1, 10 , 1, 1, 3, 4)

0.36787944117144233

Finding sigma 

In [ ]:
def variance_of_correlation(WV, A, n, m, ax, ay, dl=1):

    """
        Returns <VV> for correlation 2D  
        
        Args:
            WV (array) : Array of dim (2X2XnXm, 1) representing wavefunction 
            A (float) : normalizing ocnstant for correlation function
            dl (float) : decay length of correlation function 
        
    """

    sum = 0 

    for i in range(m*n):

        for j in range(m*n):

            sum += sum + abs(WV[i])**2*abs(WV[j])**2*Correlation_2D(A, n, ax, ay, i, j, dl)

    return sum